# Phase 1 — Data Ingestion

Downloads and merges all three data sources into a single `master.parquet`.

**Run this notebook first** — every other notebook depends on its output.

> **Pipeline upgrade (v2):** The ingestion pipeline now fetches the **full German
> generation mix**, including minor dispatchable renewables (Biomass, Run-of-River,
> Pumped Storage Generation, Other Renewables). It also dynamically calculates a
> **`carbon_intensity_g_kwh`** feature (g CO₂-eq/kWh) based on the live generation
> mix, enabling downstream environmental forecasting alongside the existing load
> and renewable-energy targets.

### Install dependencies
```bash
pip install pandas pyarrow requests tqdm meteostat holidays seaborn pvlib
```

### Data sources
| Source | What | License |
|--------|------|---------|
| OPSD | German load + solar + wind (hourly, 2015–2020) | MIT / CC BY 4.0 |
| SMARD | Official Bundesnetzagentur data — full generation mix (2015–present) | dl-de/by-2-0 |
| Meteostat + DWD | Weather — 20 DE stations + national composites | CC BY-NC 4.0 / DWD Open |


In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')
RESULTS   = Path('../results')

for d in [RAW, PROCESSED, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 110
print('Directories ready:')
for d in [RAW, PROCESSED, RESULTS]:
    print(f'  {d.resolve()}')

## 1. OPSD — Primary dataset

Downloads ~150 MB CSV from the OPSD data portal. Cached after first run.

OPSD covers 2015 → Oct 2020. The dataset ends there — the raw output from
`load_opsd()` will show NaN values for all target columns after that date.
The SMARD backfill that extends the series through to the present is applied
later inside `build_master()` in Section 4.

In [ ]:
from src.ingest.opsd import load_opsd

opsd = load_opsd(RAW, PROCESSED, force=False)
print(f'Shape: {opsd.shape}')
print(f'Range: {opsd.index.min().date()} -> {opsd.index.max().date()}')
print(f'\nMissing values (%):')
print((opsd.isna().mean() * 100).round(2).to_string())
opsd.tail(3)

## 2. SMARD — Full generation mix

Fetches all 14 generation and consumption series from the official SMARD API
(Bundesnetzagentur). First run takes **10–25 minutes** (14 filters × many
weekly chunks). All chunks are cached in `data/raw/` — subsequent runs are
instant.

The SMARD chunk cache is shared with the OPSD backfill above, so any chunks
already downloaded will not be re-fetched.

> ⚠️ **Cache rebuild required on first run after the v2 upgrade.**
> The pipeline now fetches additional SMARD generation series and computes
> `carbon_intensity_g_kwh` inside `build_master()`. Old Parquet caches do **not**
> contain these columns. Pass `force=True` to `load_smard()`, `load_meteostat()`,
> and `build_master()` on your first run to rebuild the caches with the new data.
> You can revert to `force=False` once the caches are up to date.


In [ ]:
from src.ingest.smard import load_smard

START = '2018-01-01'   # change to '2020-01-01' for a faster first run

smard = load_smard(RAW, PROCESSED, start=START, force=False  # change to force=True on first run after v2 upgrade)
print(f'Shape: {smard.shape}')
print(f'Range: {smard.index.min().date()} -> {smard.index.max().date()}')
print(f'\nColumns and NaN rates:')
for col in smard.columns:
    print(f'  {col:<40} {smard[col].isna().mean()*100:5.1f}% NaN')
smard.tail(3)

In [ ]:
# Cross-check: OPSD vs SMARD series comparison.
# The detailed OPSD-original vs SMARD-backfill splice visualisation
# is now in Section 4 (after build_master), where the provenance flags
# written by merge.py are available.  A quick numeric summary is shown
# here instead to confirm the two sources are compatible.

opsd_valid    = opsd['load_mw'].dropna()
overlap_start = max(opsd_valid.index.min(), smard.index.min())
overlap_end   = opsd_valid.index.max()

# Pearson correlation between OPSD and SMARD load over the overlap period
overlap_opsd  = opsd['load_mw'].loc[overlap_start:overlap_end].dropna()
overlap_smard = smard['load_mwh'].reindex(overlap_opsd.index)
corr = overlap_opsd.corr(overlap_smard)
mae  = (overlap_opsd - overlap_smard).abs().mean()
print(f'OPSD vs SMARD load — overlap period: {overlap_start.date()} → {overlap_end.date()}')
print(f'  Pearson r : {corr:.4f}  (should be > 0.99)')
print(f'  MAE       : {mae:.1f} MW  (should be small)')
print(f'  OPSD rows : {len(overlap_opsd):,}')
print(f'  SMARD rows: {overlap_smard.notna().sum():,}')


## 3. Meteostat — Weather data

Downloads hourly weather for 20 German stations covering all major climate
zones (North Sea coast, Baltic, NRW, central, East, Rhine valley, Bavaria).
Falls back to direct DWD bulk CSV download if the Meteostat library call fails.

Produces per-station columns (e.g. `hamburg_temp`) and area-weighted national
composites (e.g. `de_temp`, `de_wspd`, `de_tsun`).

In [ ]:
from src.ingest.meteostat import load_meteostat

weather = load_meteostat(PROCESSED, start=START, force=False  # change to force=True on first run after v2 upgrade)
print(f'Shape: {weather.shape}')
print(f'Range: {weather.index.min().date()} -> {weather.index.max().date()}')

composite_cols = [c for c in weather.columns if c.startswith('de_')]
print(f'\nNational composite columns: {composite_cols}')
print(f'\nMissing values (composite columns, %):')
print((weather[composite_cols].isna().mean() * 100).round(2).to_string())
weather[composite_cols].tail(3)

In [ ]:
# Sunshine duration — verify continuous coverage across the full date range
if 'de_tsun' in weather.columns:
    fig, ax = plt.subplots(figsize=(14, 3))
    weather['de_tsun'].resample('D').mean().plot(ax=ax, lw=0.6, color='goldenrod')
    ax.set_title('de_tsun (sunshine min/h) — national composite')
    ax.set_ylabel('min/h')
    ax.set_xlabel('Date')
    plt.tight_layout()
    plt.savefig(RESULTS / 'de_tsun_coverage.png', bbox_inches='tight')
    plt.show()
    print(f'de_tsun NaN rate: {weather["de_tsun"].isna().mean()*100:.1f}%')
else:
    print('de_tsun was dropped (too sparse after DWD fallback). '
          'Set DROP_TSUN_IF_SPARSE=False in meteostat.py to retry.')

In [ ]:
# Overview of national composite weather variables (daily averages)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
daily = weather[composite_cols].resample('D').mean()

for ax, (col, color, title) in zip(axes, [
    ('de_temp', 'crimson',   'National avg temperature (°C)'),
    ('de_wspd', 'steelblue', 'National avg wind speed (km/h)'),
    ('de_tsun', 'goldenrod', 'National avg sunshine (min/h)'),
    ('de_prcp', 'teal',      'National avg precipitation (mm/h)'),
]):
    if col in daily.columns:
        daily[col].plot(ax=ax, color=color, lw=0.8)
    ax.set_title(title)
    ax.set_xlabel('')

axes[-1].set_xlabel('Date')
plt.suptitle('Meteostat — German national weather composites (daily avg)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'meteostat_overview.png', bbox_inches='tight')
plt.show()

## 4. Merge → master.parquet

Aligns all three sources to a clean hourly UTC index, forward-fills short
gaps (≤ 3 h), applies the SMARD backfill for target columns missing after
October 2020, and saves `master.parquet`.

In [ ]:
from src.ingest.merge import build_master

master = build_master(
    raw_dir=RAW,
    processed_dir=PROCESSED,
    start=START,
    force=False  # change to force=True on first run after v2 upgrade,
)

print(f'Master shape : {master.shape}')
print(f'Date range   : {master.index.min().date()} -> {master.index.max().date()}')
print(f'\nAll columns:')
for col in master.columns:
    nan_pct = master[col].isna().mean() * 100
    print(f'  {col:<40}  {nan_pct:5.1f}% NaN')

In [ ]:
# Visualise the OPSD-to-SMARD splice for each target column.
# merge.py writes a boolean provenance flag ({col}_from_smard) for every
# backfilled column, which lets us colour the two segments independently.

target_cols  = ['load_mw', 'solar_mw', 'wind_onshore_mw', 'wind_offshore_mw']
target_cols  = [c for c in target_cols if c in master.columns]

fig, axes = plt.subplots(len(target_cols), 1,
                         figsize=(14, 3 * len(target_cols)), sharex=True)
if len(target_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, target_cols):
    flag_col = f'{col}_from_smard'
    from_smard = master[flag_col] if flag_col in master.columns \
                 else pd.Series(False, index=master.index)

    opsd_segment  = master[col].where(~from_smard)
    smard_segment = master[col].where(from_smard)

    opsd_segment.resample('W').mean().plot(
        ax=ax, lw=0.9, color='steelblue', label='OPSD (original)')
    smard_segment.resample('W').mean().plot(
        ax=ax, lw=0.9, color='crimson', label='SMARD (backfill)')

    splice_dates = master.index[from_smard]
    if len(splice_dates):
        ax.axvline(splice_dates.min(), color='orange', lw=1.5,
                   ls='--', label=f'Splice ({splice_dates.min().date()})')

    ax.set_title(col)
    ax.set_ylabel('MW (weekly avg)')
    ax.legend(fontsize=7, ncol=3)

axes[-1].set_xlabel('Date')
plt.suptitle('Master dataset — OPSD original + SMARD backfill by target column',
             fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'opsd_backfill_splice.png', bbox_inches='tight')
plt.show()

## 5. EDA — Data quality and solar analysis

The clearsky index measures how much of the theoretical maximum solar
irradiance is actually reaching the grid at each hour. It is computed using
pvlib's Ineichen clearsky model at Germany's geographic centroid, normalised
against installed solar capacity.

In [ ]:
import pvlib

# Germany geographic centroid and approximate mean altitude
DE_LAT, DE_LON, DE_ALT = 51.2, 10.4, 300.0

# Installed solar PV capacity reference (GW) — used only for normalisation
# Conservative value to avoid capacity-factor inflation in early years
DE_SOLAR_CAPACITY_GW = 70.0


def compute_clearsky_index(master_df: pd.DataFrame) -> pd.Series:
    """
    Compute an hourly clearsky index for Germany.

    Uses pvlib to get theoretical clearsky GHI at Germany's centroid,
    then compares it against observed solar generation normalised by
    installed capacity. Result is dimensionless [0, 2] and NaN at night.
    """
    idx = master_df.index

    location  = pvlib.location.Location(DE_LAT, DE_LON, altitude=DE_ALT, tz='UTC')
    solar_pos = location.get_solarposition(idx)
    cs        = location.get_clearsky(idx, model='ineichen')
    ghi       = cs['ghi'].values.astype(np.float64)

    # Normalise clearsky GHI to a [0, 1] fraction using the 99th percentile
    ghi_p99     = np.nanpercentile(ghi[ghi > 0], 99)
    cs_fraction = ghi / ghi_p99

    # Observed solar as a capacity factor (MWh / peak_MW)
    solar_col   = 'solar_mwh_smard' if 'solar_mwh_smard' in master_df.columns else 'solar_mw'
    obs_fraction = np.clip(
        master_df[solar_col].values.astype(np.float64) / (DE_SOLAR_CAPACITY_GW * 1000.0),
        0.0, None
    )

    # Only compute during daylight hours (elevation > 0 and GHI > 1 W/m²)
    is_day = (solar_pos['elevation'].values > 0) & (ghi > 1.0)
    result = np.full(len(idx), np.nan, dtype=np.float64)
    mask   = is_day & np.isfinite(obs_fraction) & (cs_fraction > 0.01)
    result[mask] = np.clip(obs_fraction[mask] / cs_fraction[mask], 0.0, 2.0)

    return pd.Series(result.astype(np.float32), index=idx, name='clearsky_index')


master['clearsky_index'] = compute_clearsky_index(master)

daylight = master['clearsky_index'].dropna()
print(f'Clearsky index (daylight hours only):')
print(daylight.describe().round(3).to_string())

In [ ]:
# Clearsky index should show a clear diurnal cycle and seasonal pattern
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

# Two-week sample showing hourly diurnal peaks
master['clearsky_index']['2022-06-01':'2022-06-14'].plot(
    ax=axes[0], color='goldenrod', lw=0.9)
axes[0].set_title('Clearsky index — 2-week sample (diurnal peaks visible)')
axes[0].set_ylabel('Index (0=overcast, 1=clear, >1=enhancement)')
axes[0].axhline(1.0, color='gray', ls='--', lw=0.8, label='Clear sky = 1.0')
axes[0].legend(fontsize=8)

# Monthly mean showing higher values in summer
master['clearsky_index'].resample('ME').mean().plot(
    ax=axes[1], color='darkorange', lw=1.2, marker='o', ms=3)
axes[1].set_title('Clearsky index — monthly mean (higher in summer)')
axes[1].set_ylabel('Mean index')

plt.tight_layout()
plt.savefig(RESULTS / 'clearsky_index.png', bbox_inches='tight')
plt.show()

## 6. Data quality checks

In [ ]:
# Missing data heatmap — visualise gaps per month across all target columns
target_cols = [c for c in
               ['load_mw', 'solar_mw', 'wind_onshore_mw', 'wind_offshore_mw']
               if c in master.columns]

monthly_nan = (
    master[target_cols]
    .resample('ME')
    .apply(lambda x: x.isna().mean() * 100)
)

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(monthly_nan.T, ax=ax, cmap='Reds',
            cbar_kws={'label': 'NaN %'},
            linewidths=0.3, linecolor='white')
ax.set_title('Missing data (%) by month — target columns')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig(RESULTS / 'missing_data_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Weather vs target correlation matrix
# Uses pairwise complete observations so columns with different NaN windows
# still produce valid correlations
weather_composite_cols = [c for c in master.columns if c.startswith('de_')]

MAX_NAN_FRAC    = 0.50  # drop columns with more than 50% NaN
MIN_VALID_PAIRS = 100   # require at least this many overlapping rows per pair

all_cols  = [c for c in weather_composite_cols + target_cols if c in master.columns]
sub       = master[all_cols]
nan_rates = sub.isna().mean()
dropped   = nan_rates[nan_rates > MAX_NAN_FRAC].index.tolist()
if dropped:
    print(f'Dropped (>{MAX_NAN_FRAC*100:.0f}% NaN): {dropped}')
sub = sub.drop(columns=dropped)

feat_cols = [c for c in weather_composite_cols if c in sub.columns]
tgt_cols  = [c for c in target_cols            if c in sub.columns]

if feat_cols and tgt_cols:
    corr = sub.corr(min_periods=MIN_VALID_PAIRS).loc[feat_cols, tgt_cols]
    fig, ax = plt.subplots(figsize=(max(7, len(tgt_cols)*1.8), max(5, len(corr)*0.55)))
    sns.heatmap(
        corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
        linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8},
        mask=corr.isna(), annot_kws={'size': 9},
    )
    ax.set_title('Pearson correlation: weather composites vs generation targets')
    plt.tight_layout()
    plt.savefig(RESULTS / 'weather_target_correlation.png', bbox_inches='tight')
    plt.show()

In [ ]:
print('=== Target column statistics (MW) ===')
print(master[target_cols].describe().round(1).to_string())

if 'clearsky_index' in master.columns:
    print('\n=== Clearsky index statistics (daylight hours) ===')
    print(master['clearsky_index'].dropna().describe().round(3).to_string())

print(f'\nmaster.parquet: {(PROCESSED / "master.parquet").resolve()}')
print('\nPhase 1 complete. Run 02_feature_engineering.ipynb next.')